In [0]:
df = spark.read.parquet("/Volumes/retail/default/raw_parquet/ns_brands_2026-04-22.parquet")
display(df)
print(df.count())

In [0]:
from pyspark.sql.functions import current_timestamp, lit, current_date, sha2, concat_ws, col

run_id = "RUN_001"
source_system = "Postgres"

df_brands = (
    spark.read.parquet("/Volumes/retail/default/raw_parquet/ns_brands_2026-04-22.parquet")
    .withColumn("internal_id", col("internal_id").cast("string"))
    .withColumn("brand_code", col("brand_code").cast("string"))
    .withColumn("brand_name", col("brand_name").cast("string"))
    .withColumn("is_active", col("is_active").cast("string"))
    .withColumn("created_at", col("created_at").cast("string"))
)

df_brands_final = (
    df_brands
    .withColumn("etl_loaded_at", current_timestamp())
    .withColumn("source_system", lit(source_system))
    .withColumn("etl_run_id", lit(run_id))
    .withColumn("ingestion_date", current_date())
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("internal_id"),
                col("brand_code"),
                col("brand_name"),
                col("is_active"),
                col("created_at")
            ),
            256
        )
    )
    .withColumn("is_deleted", lit(False))
)

df_brands_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.bronze.ns_brands")

In [0]:
df_brands_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.bronze.ns_brands")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, current_date, sha2, concat_ws, col

run_id = "RUN_001"
source_system = "Postgres"
base_path = "/Volumes/retail/default/raw_parquet"

table_configs = [
    {
        "table_name": "ns_customers",
        "file_name": "ns_customers_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_customers",
        "cast_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_items",
        "file_name": "ns_items_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_items",
        "cast_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_orders",
        "file_name": "ns_sales_orders_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_sales_orders",
        "cast_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date", "order_status",
            "sales_channel", "payment_status", "currency_code", "external_ref_num",
            "integration_source", "order_total", "created_at"
        ],
        "hash_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date", "order_status",
            "sales_channel", "payment_status", "currency_code", "external_ref_num",
            "integration_source", "order_total", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_order_lines",
        "file_name": "ns_sales_order_lines_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_sales_order_lines",
        "cast_cols": [
            "internal_id", "sales_order_internal_id", "line_num", "item_internal_id",
            "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "hash_cols": [
            "internal_id", "sales_order_internal_id", "line_num", "item_internal_id",
            "quantity", "unit_rate", "line_amount", "created_at"
        ]
    },
    {
        "table_name": "ns_shipments",
        "file_name": "ns_shipments_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_shipments",
        "cast_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name",
            "shipment_date", "delivery_date", "shipment_status", "tracking_number", "created_at"
        ],
        "hash_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name",
            "shipment_date", "delivery_date", "shipment_status", "tracking_number", "created_at"
        ]
    }
]

def load_to_bronze(file_name, target_table, cast_cols, hash_cols):
    file_path = f"{base_path}/{file_name}"
    df = spark.read.parquet(file_path)

    for c in cast_cols:
        df = df.withColumn(c, col(c).cast("string"))

    df_final = (
        df.withColumn("etl_loaded_at", current_timestamp())
          .withColumn("source_system", lit(source_system))
          .withColumn("etl_run_id", lit(run_id))
          .withColumn("ingestion_date", current_date())
          .withColumn(
              "record_hash",
              sha2(concat_ws("||", *[col(c) for c in hash_cols]), 256)
          )
          .withColumn("is_deleted", lit(False))
    )

    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    print(f"{target_table}: {spark.table(target_table).count()} rows loaded")

for cfg in table_configs:
    load_to_bronze(
        cfg["file_name"],
        cfg["target_table"],
        cfg["cast_cols"],
        cfg["hash_cols"]
    )

# Silver Cleaned Tables

In [0]:
from pyspark.sql.functions import col, trim, lower, when
df_brand_silver = (
    spark.table("retail.bronze.ns_brands").select(
        col("internal_id").cast("int").alias("internal_id"),
        trim(col("brand_code")).alias("brand_code"),
        trim(col("brand_name")).alias("brand_name"),
        when(lower(trim(col("is_active"))).isin("true", "1", "yes"), True).otherwise(False).alias("is_active"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_brand_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_brands")

In [0]:
from pyspark.sql.functions import col, trim, lower, when

df_customers_silver = (
    spark.table("retail.bronze.ns_customers")
    .select(
        col("internal_id").cast("int").alias("internal_id"),
        trim(col("entity_id")).alias("entity_id"),
        trim(col("company_name")).alias("company_name"),
        trim(col("customer_type")).alias("customer_type"),
        trim(col("email")).alias("email"),
        trim(col("phone")).alias("phone"),
        trim(col("subsidiary")).alias("subsidiary"),
        trim(col("currency_code")).alias("currency_code"),
        trim(col("terms_name")).alias("terms_name"),
        trim(col("vat_number")).alias("vat_number"),
        trim(col("billing_address1")).alias("billing_address1"),
        trim(col("billing_city")).alias("billing_city"),
        trim(col("billing_state")).alias("billing_state"),
        trim(col("billing_country")).alias("billing_country"),
        trim(col("shipping_address1")).alias("shipping_address1"),
        trim(col("shipping_city")).alias("shipping_city"),
        trim(col("shipping_state")).alias("shipping_state"),
        trim(col("shipping_country")).alias("shipping_country"),
        when(lower(trim(col("is_active"))).isin("true", "1", "yes"), True).otherwise(False).alias("is_active"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_customers")

In [0]:
from pyspark.sql.functions import col, trim, lower, when

df_items_silver = (
    spark.table("retail.bronze.ns_items")
    .select(
        col("internal_id").cast("int").alias("internal_id"),
        trim(col("item_id")).alias("item_id"),
        trim(col("item_name")).alias("item_name"),
        trim(col("item_type")).alias("item_type"),
        trim(col("category")).alias("category"),
        trim(col("subcategory")).alias("subcategory"),
        col("brand_internal_id").cast("int").alias("brand_internal_id"),
        col("base_price").cast("decimal(18,2)").alias("base_price"),
        col("cost_price").cast("decimal(18,2)").alias("cost_price"),
        when(lower(trim(col("is_active"))).isin("true", "1", "yes"), True).otherwise(False).alias("is_active"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_items")

In [0]:
from pyspark.sql.functions import col, trim

df_orders_silver = (
    spark.table("retail.bronze.ns_sales_orders")
    .select(
        col("internal_id").cast("int").alias("internal_id"),
        trim(col("tran_id")).alias("tran_id"),
        col("customer_internal_id").cast("int").alias("customer_internal_id"),
        col("order_date").cast("timestamp").alias("order_date"),
        trim(col("order_status")).alias("order_status"),
        trim(col("sales_channel")).alias("sales_channel"),
        trim(col("payment_status")).alias("payment_status"),
        trim(col("currency_code")).alias("currency_code"),
        trim(col("external_ref_num")).alias("external_ref_num"),
        trim(col("integration_source")).alias("integration_source"),
        col("order_total").cast("decimal(18,2)").alias("order_total"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_sales_orders")

In [0]:
from pyspark.sql.functions import col, trim

df_lines_silver = (
    spark.table("retail.bronze.ns_sales_order_lines")
    .select(
        col("internal_id").cast("int").alias("internal_id"),
        col("sales_order_internal_id").cast("int").alias("sales_order_internal_id"),
        col("line_num").cast("int").alias("line_num"),
        col("item_internal_id").cast("int").alias("item_internal_id"),
        col("quantity").cast("int").alias("quantity"),
        col("unit_rate").cast("decimal(18,2)").alias("unit_rate"),
        col("line_amount").cast("decimal(18,2)").alias("line_amount"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_lines_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_sales_order_lines")

In [0]:
from pyspark.sql.functions import col, trim

df_shipments_silver = (
    spark.table("retail.bronze.ns_shipments")
    .select(
        col("internal_id").cast("int").alias("internal_id"),
        trim(col("shipment_number")).alias("shipment_number"),
        col("sales_order_internal_id").cast("int").alias("sales_order_internal_id"),
        trim(col("warehouse_name")).alias("warehouse_name"),
        col("shipment_date").cast("timestamp").alias("shipment_date"),
        col("delivery_date").cast("timestamp").alias("delivery_date"),
        trim(col("shipment_status")).alias("shipment_status"),
        trim(col("tracking_number")).alias("tracking_number"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("etl_loaded_at").cast("timestamp").alias("etl_loaded_at"),
        trim(col("source_system")).alias("source_system"),
        trim(col("etl_run_id")).alias("etl_run_id"),
        col("ingestion_date").cast("date").alias("ingestion_date"),
        col("record_hash"),
        col("is_deleted").cast("boolean").alias("is_deleted")
    )
)

df_shipments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.silver.ns_shipments")

# Gold Dimenssion Tables

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

df_dim_brand = (
    spark.table("retail.silver.ns_brands")
    .select(
        col("internal_id").alias("brand_internal_id"),
        col("brand_code"),
        col("brand_name"),
        col("is_active")
    )
    .withColumn("brand_key", monotonically_increasing_id())
)

df_dim_brand.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.gold.dim_brand")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

df_dim_customer = (
    spark.table("retail.silver.ns_customers")
    .select(
        col("internal_id").alias("customer_internal_id"),
        col("entity_id"),
        col("company_name"),
        col("customer_type"),
        col("email"),
        col("phone"),
        col("subsidiary"),
        col("currency_code"),
        col("terms_name"),
        col("vat_number"),
        col("billing_city"),
        col("billing_country"),
        col("shipping_city"),
        col("shipping_country"),
        col("is_active")
    )
    .withColumn("customer_key", monotonically_increasing_id())
)

df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.gold.dim_customer")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

df_dim_customer = (
    spark.table("retail.silver.ns_customers")
    .select(
        col("internal_id").alias("customer_internal_id"),
        col("entity_id"),
        col("company_name"),
        col("customer_type"),
        col("email"),
        col("phone"),
        col("subsidiary"),
        col("currency_code"),
        col("terms_name"),
        col("vat_number"),
        col("billing_city"),
        col("billing_country"),
        col("shipping_city"),
        col("shipping_country"),
        col("is_active")
    )
    .withColumn("customer_key", monotonically_increasing_id())
    .select(
        "customer_key",
        "customer_internal_id",
        "entity_id",
        "company_name",
        "customer_type",
        "email",
        "phone",
        "subsidiary",
        "currency_code",
        "terms_name",
        "vat_number",
        "billing_city",
        "billing_country",
        "shipping_city",
        "shipping_country",
        "is_active"
    )
)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

df_items = spark.table("retail.silver.ns_items").alias("i")
df_brands = spark.table("retail.gold.dim_brand").alias("b")

df_dim_item = (
    df_items.join(
        df_brands,
        col("i.brand_internal_id") == col("b.brand_internal_id"),
        "left"
    )
    .select(
        col("i.internal_id").alias("item_internal_id"),
        col("i.item_id"),
        col("i.item_name"),
        col("i.item_type"),
        col("i.category"),
        col("i.subcategory"),
        col("i.brand_internal_id"),
        col("b.brand_key"),
        col("b.brand_name"),
        col("i.base_price"),
        col("i.cost_price"),
        col("i.is_active")
    )
    .withColumn("item_key", monotonically_increasing_id())
    .select(
        "item_key",
        "item_internal_id",
        "item_id",
        "item_name",
        "item_type",
        "category",
        "subcategory",
        "brand_internal_id",
        "brand_key",
        "brand_name",
        "base_price",
        "cost_price",
        "is_active"
    )
)

df_dim_item.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.gold.dim_item")

In [0]:
from pyspark.sql.functions import (
    col, to_date, date_format, dayofmonth, month, year,
    quarter, weekofyear, date_format
)

df_dim_date = (
    spark.table("retail.silver.ns_sales_orders")
    .select(to_date(col("order_date")).alias("full_date"))
    .distinct()
    .filter(col("full_date").isNotNull())
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("month_num", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("year_num", year(col("full_date")))
    .withColumn("quarter_num", quarter(col("full_date")))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .select(
        "date_key",
        "full_date",
        "day_of_month",
        "month_num",
        "month_name",
        "year_num",
        "quarter_num",
        "week_of_year"
    )
)

df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.gold.dim_date")

# Fact Table

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, to_date, coalesce, lit

df_orders = spark.table("retail.silver.ns_sales_orders").alias("o")
df_lines = spark.table("retail.silver.ns_sales_order_lines").alias("l")
df_customer = spark.table("retail.gold.dim_customer").alias("c")
df_item = spark.table("retail.gold.dim_item").alias("i")
df_date = spark.table("retail.gold.dim_date").alias("d")

df_fact_sales = (
    df_lines
    .join(
        df_orders,
        col("l.sales_order_internal_id") == col("o.internal_id"),
        "inner"
    )
    .join(
        df_customer,
        col("o.customer_internal_id") == col("c.customer_internal_id"),
        "left"
    )
    .join(
        df_item,
        col("l.item_internal_id") == col("i.item_internal_id"),
        "left"
    )
    .join(
        df_date,
        to_date(col("o.order_date")) == col("d.full_date"),
        "left"
    )
    .select(
        monotonically_increasing_id().alias("sales_key"),
        col("o.internal_id").alias("sales_order_internal_id"),
        col("o.tran_id"),
        col("l.internal_id").alias("sales_order_line_internal_id"),
        col("d.date_key"),
        col("o.order_date"),
        col("c.customer_key"),
        col("c.customer_internal_id"),
        col("i.item_key"),
        col("i.item_internal_id"),
        col("i.brand_key"),
        col("l.line_num"),
        col("l.quantity"),
        col("l.unit_rate").alias("unit_price"),
        col("l.line_amount"),
        col("o.order_status"),
        col("o.sales_channel"),
        col("o.payment_status"),
        col("o.currency_code"),
        col("o.integration_source"),
        col("o.etl_loaded_at"),
        col("o.etl_run_id"),
        col("o.source_system")
    )
)

df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.gold.fact_sales")